# Milestone 1: Data Exploration and Preprocessing

The purpose of this notebook is to:

- load and inspect small samples of the Amazon Reviews 2023 dataset
- examine both **review** and **metadata** records
- identify the most useful fields for retrieval
- justify preprocessing choices
- create a compact cleaned dataset for downstream **BM25** and **semantic retrieval**

For this milestone, the analysis begins with two sampled categories:

- **All_Beauty**
- **Health_and_Personal_Care**

After comparing them, one category is selected for the first retrieval system.

This notebook is intentionally focused and lightweight. The milestone instructions recommend using a small subset such as the first 100–200 entries for EDA rather than loading full files. :contentReference[oaicite:2]{index=2}


## Imports

In [10]:
from pathlib import Path
import json
import re
from typing import Optional

import pandas as pd

In [11]:
def find_repo_root(max_up: int = 6) -> Path:
    p = Path.cwd()
    for _ in range(max_up):
        if (p / ".git").exists() or (p / "README.md").exists():
            return p
        p = p.parent
    return Path.cwd()

repo_root = find_repo_root()
repo_root

WindowsPath('c:/Users/ruthy/assignments/block6/group_575/DSCI_575_project_omo001_deepray')

## Locate sample files

This notebook assumes small sampled JSONL files are stored under `data/processed/` which is the case with out project structure.

For EDA, both **review** and **metadata** samples should be inspected. This is important because retrieval design depends on understanding:
- what the review text contains
- what useful contextual product metadata is available
- how those two sources can be combined into a better retrieval document

In [12]:
sample_dir = repo_root / "data" / "processed"

paths = {
    "all_beauty_reviews": sample_dir / "sample_All_Beauty.jsonl",
    "all_beauty_meta": sample_dir / "sample_meta_All_Beauty.jsonl",
    "health_reviews": sample_dir / "sample_Health_and_Personal_Care.jsonl",
    "health_meta": sample_dir / "sample_meta_Health_and_Personal_Care.jsonl",
}

for name, path in paths.items():
    print(f"{name}: {path} | exists={path.exists()}")

all_beauty_reviews: c:\Users\ruthy\assignments\block6\group_575\DSCI_575_project_omo001_deepray\data\processed\sample_All_Beauty.jsonl | exists=True
all_beauty_meta: c:\Users\ruthy\assignments\block6\group_575\DSCI_575_project_omo001_deepray\data\processed\sample_meta_All_Beauty.jsonl | exists=True
health_reviews: c:\Users\ruthy\assignments\block6\group_575\DSCI_575_project_omo001_deepray\data\processed\sample_Health_and_Personal_Care.jsonl | exists=True
health_meta: c:\Users\ruthy\assignments\block6\group_575\DSCI_575_project_omo001_deepray\data\processed\sample_meta_Health_and_Personal_Care.jsonl | exists=True


In [35]:
for name, df in _dfs.items():
    if df.empty:
        continue
    print(f"\n{name} sample record:")
    print(df.iloc[0].to_dict())


All_Beauty sample record:
{'rating': 5, 'title': 'Such a lovely scent but not overpowering.', 'text': "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!", 'images': [], 'asin': 'B00YQ6X8EO', 'parent_asin': 'B00YQ6X8EO', 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ', 'timestamp': Timestamp('2020-05-05 14:08:48.923000'), 'helpful_vote': 0, 'verified_purchase': True}

Health_and_Personal_Care sample record:
{'rating': 4, 'title': '12 mg is 12 on the periodic table people! Mg for magnesium', 'text': 'This review is more to clarify someone else’s review bc they didn’t understand understand the labeling!  It shows 1000mg as advertised & another little label says 12mg bc 12 is on the periodic table for magnesium!  I realize not everyone takes chemis

In [13]:
def load_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing file: {path}")
        return pd.DataFrame()

    try:
        return pd.read_json(path, lines=True)
    except ValueError:
        records = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        return pd.DataFrame(records)

In [15]:
all_beauty_reviews = load_jsonl(paths["all_beauty_reviews"])
all_beauty_meta = load_jsonl(paths["all_beauty_meta"])
health_reviews = load_jsonl(paths["health_reviews"])
health_meta = load_jsonl(paths["health_meta"])

datasets = {
    "All_Beauty reviews": all_beauty_reviews,
    "All_Beauty metadata": all_beauty_meta,
    "Health_and_Personal_Care reviews": health_reviews,
    "Health_and_Personal_Care metadata": health_meta,
}

for name, df in datasets.items():
    print(f"{name}: shape={df.shape}")

All_Beauty reviews: shape=(200, 10)
All_Beauty metadata: shape=(200, 14)
Health_and_Personal_Care reviews: shape=(200, 10)
Health_and_Personal_Care metadata: shape=(200, 14)


## Dataset overview

The Amazon Reviews 2023 dataset contains separate files for:
- **reviews**, which contain user-written review text and ratings
- **metadata**, which contain product-level attributes such as title, categories, descriptions, and features

For this milestone, these two sources are examined together because retrieval quality may improve when review text is enriched with product metadata. The milestone instructions explicitly expect the notebook to include an overview of fields, dataset size, example records, and field selection for retrieval. :contentReference[oaicite:3]{index=3}

In [16]:
overview_rows = []

for name, df in datasets.items():
    overview_rows.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "column_names": ", ".join(df.columns.tolist())
    })

overview_df = pd.DataFrame(overview_rows)
overview_df

,dataset,rows,columns,column_names
0,All_Beauty reviews,200,10,"rating, title, text, images, asin, parent_asin..."
1,All_Beauty metadata,200,14,"main_category, title, average_rating, rating_n..."
2,Health_and_Personal_Care reviews,200,10,"rating, title, text, images, asin, parent_asin..."
3,Health_and_Personal_Care metadata,200,14,"main_category, title, average_rating, rating_n..."


## Inspect sample records

Sample records are printed below to understand the structure of both review files and metadata files.

This step helps determine:
- which columns are consistently populated
- which fields are likely useful for retrieval
- whether metadata fields should be merged with reviews during document construction

In [17]:
for name, df in datasets.items():
    if not df.empty:
        print(f"\n{name} — first record")
        display(df.head(1))


All_Beauty reviews — first record


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05 14:08:48.923,0,True



All_Beauty metadata — first record


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,10,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Howard Products,[],{'Package Dimensions': '7.1 x 5.5 x 3 inches; ...,B01CUPMQZE,NaN



Health_and_Personal_Care reviews — first record


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4,12 mg is 12 on the periodic table people! Mg f...,This review is more to clarify someone else’s ...,[],B07TDSJZMR,B07TDSJZMR,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2020-02-06 00:49:35.902,3,True



Health_and_Personal_Care metadata — first record


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Health & Personal Care,Silicone Bath Body Brush Exfoliator Shower Bac...,3.9,7,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Rzoeox,[],{'Package Dimensions': '15 x 3.3 x 1.5 inches;...,B07V346GZH,NaN


## Quick field inspection

The next step is to inspect important columns and missingness patterns. This helps identify fields that are useful and practical for retrieval.

In [18]:
def summarize_columns(df: pd.DataFrame) -> pd.DataFrame:
    summary = pd.DataFrame({
        "column": df.columns,
        "non_null_count": [df[col].notna().sum() for col in df.columns],
        "null_count": [df[col].isna().sum() for col in df.columns],
        "dtype": [str(df[col].dtype) for col in df.columns],
    })
    summary["non_null_pct"] = (summary["non_null_count"] / len(df)).round(3) if len(df) else 0
    return summary.sort_values("non_null_count", ascending=False).reset_index(drop=True)

for name, df in datasets.items():
    if not df.empty:
        print(f"\n{name}")
        display(summarize_columns(df))


All_Beauty reviews


,column,non_null_count,null_count,dtype,non_null_pct
0,rating,200,0,int64,1.0
1,title,200,0,str,1.0
2,text,200,0,str,1.0
3,images,200,0,object,1.0
4,asin,200,0,str,1.0
5,parent_asin,200,0,str,1.0
6,user_id,200,0,str,1.0
7,timestamp,200,0,datetime64[ms],1.0
8,helpful_vote,200,0,int64,1.0
9,verified_purchase,200,0,bool,1.0



All_Beauty metadata


,column,non_null_count,null_count,dtype,non_null_pct
0,main_category,200,0,str,1.000
1,title,200,0,str,1.000
2,average_rating,200,0,float64,1.000
3,rating_number,200,0,int64,1.000
4,features,200,0,object,1.000
5,description,200,0,object,1.000
6,images,200,0,object,1.000
7,videos,200,0,object,1.000
8,details,200,0,object,1.000
9,categories,200,0,object,1.000



Health_and_Personal_Care reviews


,column,non_null_count,null_count,dtype,non_null_pct
0,rating,200,0,int64,1.0
1,title,200,0,str,1.0
2,text,200,0,str,1.0
3,images,200,0,object,1.0
4,asin,200,0,str,1.0
5,parent_asin,200,0,str,1.0
6,user_id,200,0,str,1.0
7,timestamp,200,0,datetime64[ms],1.0
8,helpful_vote,200,0,int64,1.0
9,verified_purchase,200,0,bool,1.0



Health_and_Personal_Care metadata


,column,non_null_count,null_count,dtype,non_null_pct
0,main_category,200,0,str,1.00
1,title,200,0,str,1.00
2,average_rating,200,0,float64,1.00
3,rating_number,200,0,int64,1.00
4,features,200,0,object,1.00
5,description,200,0,object,1.00
6,images,200,0,object,1.00
7,videos,200,0,object,1.00
8,details,200,0,object,1.00
9,categories,200,0,object,1.00


## Candidate retrieval fields

For retrieval, not every field is equally useful. The most important fields are those that contribute either:

1. meaningful searchable text, or  
2. useful metadata for result display and filtering.

The review file is expected to provide the main document text, while the metadata file may provide supporting context such as product title, categories, or description.

In [20]:
candidate_columns = [
    "asin", "parent_asin", "title", "text", "rating", "average_rating",
    "description", "features", "categories", "price", "store", "details"
]

for name, df in datasets.items():
    if df.empty:
        continue
    
    available = [col for col in candidate_columns if col in df.columns]
    print(f"\n{name} candidate columns:")
    print(available)


All_Beauty reviews candidate columns:
['asin', 'parent_asin', 'title', 'text', 'rating']

All_Beauty metadata candidate columns:
['parent_asin', 'title', 'average_rating', 'description', 'features', 'categories', 'price', 'store', 'details']

Health_and_Personal_Care reviews candidate columns:
['asin', 'parent_asin', 'title', 'text', 'rating']

Health_and_Personal_Care metadata candidate columns:
['parent_asin', 'title', 'average_rating', 'description', 'features', 'categories', 'price', 'store', 'details']


## Category comparison and milestone choice

Two categories were sampled for initial inspection:

- **All_Beauty**
- **Health_and_Personal_Care**

Both are plausible retrieval domains, but for Milestone 1 only one category is needed. The milestone requires at least one category, not multiple categories. :contentReference[oaicite:4]{index=4}

The final category should be:
- manageable in size
- easy to interpret
- suitable for natural-language product search queries

After comparing the sampled review and metadata files, **All_Beauty** is selected as the main category for Milestone 1.

This choice is motivated by three practical considerations:

1. **Manageable scope**: using one category keeps the retrieval pipeline simpler and easier to interpret for the first milestone.
2. **Natural query fit**: beauty products support realistic keyword and intent-based user queries such as “moisturizer for dry skin” or “long-lasting lipstick”.
3. **Good retrieval contrast**: this category should contain cases where both lexical matching and semantic similarity can be meaningfully compared.

The analysis below therefore proceeds with **All_Beauty**.

In [21]:
reviews_df = all_beauty_reviews.copy()
meta_df = all_beauty_meta.copy()

print("All_Beauty reviews shape:", reviews_df.shape)
print("All_Beauty metadata shape:", meta_df.shape)

All_Beauty reviews shape: (200, 10)
All_Beauty metadata shape: (200, 14)


## Inspect selected category in more detail

In [22]:
display(reviews_df.head(3))
display(meta_df.head(3))

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05 14:08:48.923,0,True
1,4,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-04 18:10:55.070,1,True
2,5,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,2020-05-16 21:41:06.052,2,True


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,10,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Howard Products,[],{'Package Dimensions': '7.1 x 5.5 x 3 inches; ...,B01CUPMQZE,NaN
1,All Beauty,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,3,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Yes To,[],"{'Item Form': 'Powder', 'Skin Type': 'Acne Pro...",B076WQZGPM,NaN
2,All Beauty,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,26,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Levine Health Products,[],{'Manufacturer': 'Levine Health Products'},B000B658RI,NaN


## Join strategy

The review and metadata files are linked using the product identifier (`asin` where available).

For retrieval, this is useful because:
- the **review text** captures user opinions and experience
- the **metadata** adds product context such as title, description, and categories

Combining these produces richer retrieval documents than using review text alone.

In [23]:
common_join_cols = [col for col in ["asin", "parent_asin"] if col in reviews_df.columns and col in meta_df.columns]
common_join_cols

['parent_asin']

In [24]:
join_col = common_join_cols[0] if common_join_cols else None
print("Join column:", join_col)

Join column: parent_asin


In [25]:
if join_col is not None:
    merged_df = reviews_df.merge(
        meta_df,
        on=join_col,
        how="left",
        suffixes=("_review", "_meta")
    )
else:
    merged_df = reviews_df.copy()

print("Merged shape:", merged_df.shape)
display(merged_df.head(2))

Merged shape: (200, 23)


,rating,title_review,text,images_review,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,...,rating_number,features,description,price,images_meta,videos,store,categories,details,bought_together
0,5,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05 14:08:48.923,0,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-04 18:10:55.070,1,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Field selection for retrieval

The retrieval dataset should be compact but informative. The selected fields below are intended to support both ranking and user-facing result display.

### Selected fields

- **doc_id**: unique identifier for each retrieval document
- **asin**: product identifier
- **title**: product title for context and display
- **review_text**: the main review body, which is the core retrieval content
- **rating**: review score, useful for display and optional filtering
- **description / features / categories**: metadata fields that can enrich the retrieval text when present

### Justification

The milestone asks for a clear selection and justification of fields used for retrieval. :contentReference[oaicite:5]{index=5}

For this project:
- the **review text** is the main evidence a retriever should search over
- the **title** helps with product identification and short queries
- **metadata fields** provide additional product semantics that may help semantic retrieval
- **rating** is not a ranking signal here, but it improves result presentation in the app

In [27]:
def first_existing(df: pd.DataFrame, candidates: list[str], default=""):
    for col in candidates:
        if col in df.columns:
            return df[col]
    return pd.Series([default] * len(df), index=df.index)

retrieval_df = pd.DataFrame({
    "asin": first_existing(merged_df, ["asin", "parent_asin"], default=""),
    "title": first_existing(merged_df, ["title_meta", "title", "title_review"], default=""),
    "review_text": first_existing(merged_df, ["text", "review_text", "text_review"], default=""),
    "rating": first_existing(merged_df, ["rating", "overall"], default=0),
    "description": first_existing(merged_df, ["description", "description_meta"], default=""),
    "features": first_existing(merged_df, ["features", "features_meta"], default=""),
    "categories": first_existing(merged_df, ["categories", "categories_meta"], default="")
})

retrieval_df.head()

,asin,title,review_text,rating,description,features,categories
0,B00YQ6X8EO,NaN,This spray is really nice. It smells really go...,5,NaN,NaN,NaN
1,B081TJ8YS3,NaN,"This product does what I need it to do, I just...",4,NaN,NaN,NaN
2,B07PNNCSP9,NaN,"Smells good, feels great!",5,NaN,NaN,NaN
3,B09JS339BZ,NaN,Felt synthetic,1,NaN,NaN,NaN
4,B08BZ63GMJ,NaN,Love it,5,NaN,NaN,NaN


In [28]:
retrieval_df["doc_id"] = (
    retrieval_df["asin"].astype(str).fillna("")
    + "_"
    + retrieval_df.index.astype(str)
)

retrieval_df = retrieval_df[
    ["doc_id", "asin", "title", "review_text", "rating", "description", "features", "categories"]
].copy()

retrieval_df.head()

,doc_id,asin,title,review_text,rating,description,features,categories
0,B00YQ6X8EO_0,B00YQ6X8EO,NaN,This spray is really nice. It smells really go...,5,NaN,NaN,NaN
1,B081TJ8YS3_1,B081TJ8YS3,NaN,"This product does what I need it to do, I just...",4,NaN,NaN,NaN
2,B07PNNCSP9_2,B07PNNCSP9,NaN,"Smells good, feels great!",5,NaN,NaN,NaN
3,B09JS339BZ_3,B09JS339BZ,NaN,Felt synthetic,1,NaN,NaN,NaN
4,B08BZ63GMJ_4,B08BZ63GMJ,NaN,Love it,5,NaN,NaN,NaN


## Text preprocessing decisions

The preprocessing here is intentionally minimal.

### Goals
- preserve useful lexical information for **BM25**
- preserve semantic meaning for **embedding-based retrieval**
- remove obvious noise
- avoid aggressive normalization too early

### Applied preprocessing
- lowercase text
- remove HTML tags
- remove URLs
- normalize whitespace
- convert list-like metadata fields into readable text
- combine title, metadata, and review text into one retrieval field

### Not applied at this stage
- stemming
- lemmatization
- stopword removal
- heavy punctuation stripping

These more aggressive steps are avoided because they may remove useful signal, especially for semantic retrieval and product-specific phrases.

In [29]:
def stringify_value(x) -> str:
    if isinstance(x, list):
        return " ".join(str(item) for item in x)
    if isinstance(x, dict):
        return " ".join(f"{k}: {v}" for k, v in x.items())
    if pd.isna(x):
        return ""
    return str(x)

def simple_clean(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)                  # remove HTML tags
    text = re.sub(r"http\S+|www\.\S+", " ", text)         # remove URLs
    text = re.sub(r"\s+", " ", text)                      # normalize whitespace
    return text.strip()

In [30]:
for col in ["title", "review_text", "description", "features", "categories"]:
    retrieval_df[col] = retrieval_df[col].apply(stringify_value)

retrieval_df["combined_text"] = (
    "title: " + retrieval_df["title"].fillna("") + " "
    + "description: " + retrieval_df["description"].fillna("") + " "
    + "features: " + retrieval_df["features"].fillna("") + " "
    + "categories: " + retrieval_df["categories"].fillna("") + " "
    + "review: " + retrieval_df["review_text"].fillna("")
)

retrieval_df["text_clean"] = retrieval_df["combined_text"].apply(simple_clean)

retrieval_df[["doc_id", "text_clean"]].head()

,doc_id,text_clean
0,B00YQ6X8EO_0,title: description: features: categories: revi...
1,B081TJ8YS3_1,title: description: features: categories: revi...
2,B07PNNCSP9_2,title: description: features: categories: revi...
3,B09JS339BZ_3,title: description: features: categories: revi...
4,B08BZ63GMJ_4,title: description: features: categories: revi...


## Inspect cleaned retrieval text

The cleaned retrieval text is inspected below to verify that:
- key content has been preserved
- metadata is readable
- no obvious formatting noise remains

In [31]:
display(retrieval_df[["doc_id", "title", "review_text", "text_clean"]].head(5))

,doc_id,title,review_text,text_clean
0,B00YQ6X8EO_0,,This spray is really nice. It smells really go...,title: description: features: categories: revi...
1,B081TJ8YS3_1,,"This product does what I need it to do, I just...",title: description: features: categories: revi...
2,B07PNNCSP9_2,,"Smells good, feels great!",title: description: features: categories: revi...
3,B09JS339BZ_3,,Felt synthetic,title: description: features: categories: revi...
4,B08BZ63GMJ_4,,Love it,title: description: features: categories: revi...


## Basic quality checks

In [32]:
quality_checks = pd.DataFrame({
    "metric": [
        "number_of_documents",
        "missing_title",
        "missing_review_text",
        "empty_text_clean"
    ],
    "value": [
        len(retrieval_df),
        retrieval_df["title"].eq("").sum(),
        retrieval_df["review_text"].eq("").sum(),
        retrieval_df["text_clean"].eq("").sum()
    ]
})

quality_checks

,metric,value
0,number_of_documents,200
1,missing_title,199
2,missing_review_text,0
3,empty_text_clean,0


## Final dataset for Milestone 1

The cleaned retrieval dataset keeps the following compact fields:

- `doc_id`
- `asin`
- `title`
- `rating`
- `text_clean`

This structure is sufficient for:
- BM25 indexing
- embedding generation
- top-k retrieval
- app result display

Keeping the dataset compact reduces indexing overhead while preserving the information needed for retrieval experiments.

In [33]:
final_df = retrieval_df[["doc_id", "asin", "title", "rating", "text_clean"]].copy()
final_df.head()

,doc_id,asin,title,rating,text_clean
0,B00YQ6X8EO_0,B00YQ6X8EO,,5,title: description: features: categories: revi...
1,B081TJ8YS3_1,B081TJ8YS3,,4,title: description: features: categories: revi...
2,B07PNNCSP9_2,B07PNNCSP9,,5,title: description: features: categories: revi...
3,B09JS339BZ_3,B09JS339BZ,,1,title: description: features: categories: revi...
4,B08BZ63GMJ_4,B08BZ63GMJ,,5,title: description: features: categories: revi...


## Export cleaned dataset

The cleaned dataset is exported for downstream retrieval scripts.

Two formats are saved:

- **Parquet**: efficient for reuse in Python pipelines
- **JSONL**: convenient for document-style processing and LangChain-compatible workflows

In [34]:
output_dir = repo_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

parquet_path = output_dir / "All_Beauty_clean.parquet"
jsonl_path = output_dir / "All_Beauty_clean.jsonl"

final_df.to_parquet(parquet_path, index=False)

final_df.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)

print("Saved:", parquet_path)
print("Saved:", jsonl_path)

Saved: c:\Users\ruthy\assignments\block6\group_575\DSCI_575_project_omo001_deepray\data\processed\All_Beauty_clean.parquet
Saved: c:\Users\ruthy\assignments\block6\group_575\DSCI_575_project_omo001_deepray\data\processed\All_Beauty_clean.jsonl


## Summary of EDA and preprocessing decisions

This notebook examined sampled review and metadata files from two candidate categories and selected **All_Beauty** for Milestone 1.

The main findings are:

- review text is the core retrieval content
- metadata such as title, description, features, and categories can enrich retrieval documents
- a compact merged document format is appropriate for both BM25 and semantic retrieval
- minimal preprocessing is preferable at this stage to preserve retrieval signal

The resulting cleaned dataset will be used in later steps to build:
- a **BM25 retriever**
- a **semantic retriever** based on embeddings
- a simple search application

This aligns with the Milestone 1 objective of preparing data for retrieval rather than full generation or RAG. :contentReference[oaicite:6]{index=6} :contentReference[oaicite:7]{index=7}

In [37]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s\.,!?\'\"-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [38]:
for name, df in clean_dfs.items():
    df['combined'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
    df['clean_text'] = df['combined'].apply(clean_text)

clean_dfs[list(clean_dfs.keys())[0]][['doc_id', 'clean_text']].head()

,doc_id,clean_text
0,B00YQ6X8EO,such a lovely scent but not overpowering. this...
1,B081TJ8YS3,works great but smells a little weird. this pr...
2,B07PNNCSP9,"yes! smells good, feels great!"
3,B09JS339BZ,synthetic feeling felt synthetic
4,B08BZ63GMJ,a love it


## Export Clean Dataset

Save processed data as parquet for reuse in retrieval tasks.

We initially conducted exploratory data analysis on both the **All_Beauty** and **Health_and_Personal_Care** categories using 200-sample subsets from each dataset. This allowed us to compare their structure, inspect the available review and metadata fields, and assess their suitability for retrieval. After this comparison, we proceeded with **All_Beauty** as the primary category for Milestone 1. This decision was made because All_Beauty provides a manageable dataset size while still offering diverse, natural-language product queries that are well suited to both keyword-based retrieval and semantic search. Focusing on a single category also keeps the retrieval pipeline simpler and more interpretable for this milestone, while still fully satisfying the project requirement of using at least one category.

In [ ]:
output_dir = repo_root / 'data' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)

for name, df in clean_dfs.items():
    out_path = output_dir / f"{name}_clean.parquet"
    df_to_save = df[['doc_id', 'title', 'clean_text', 'rating']].copy()
    df_to_save.to_parquet(out_path, index=False)
    print('Saved:', out_path)